# 04.4 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/04-encoding-categories/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

six = pd.DataFrame({"customer": ["A", "B", "C", "D", "E", "F"],
                    "country":  ["UK", "DE", "FR", "UK", "FR", "DE"],
                    "plan":     ["S", "L", "M", "M", "S", "L"]})
six

ten = pd.DataFrame({"country": ["UK", "UK", "UK", "UK", "DE", "DE", "FR", "FR", "FR", "NL"],
                    "churned": [1, 0, 0, 1, 0, 0, 1, 0, 1, 1]})

past = sales[sales["InvoiceDate"] < "2011-09-01"]
by_item = past.groupby(["Customer ID", "StockCode"])["Quantity"].sum().reset_index()
fav = by_item.sort_values("Quantity").groupby("Customer ID").tail(1).set_index("Customer ID")["StockCode"]
cust["fav_product"] = fav.reindex(cust.index)       # the product each customer bought most units of

from sklearn.model_selection import train_test_split
tr, te = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
y_tr, y_te = tr["churned"], te["churned"]
base = ["days_since_last", "orders"]
print(f"{cust['country'].nunique()} countries, {cust['fav_product'].nunique()} different favourite products")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def fit_score(A, B):
    """Scale, fit a logistic regression on A (train), return (train acc, test acc)."""
    sc = StandardScaler().fit(A)
    m = LogisticRegression(max_iter=2000).fit(sc.transform(A), y_tr)
    return m.score(sc.transform(A), y_tr), m.score(sc.transform(B), y_te)

3,317 customers, churn rate 43.6%


36 countries, 951 different favourite products


## Exercise 1

- **Label (alphabetical):** "ES" has no code. Default behaviour is an error; with `handle_unknown="use_encoded_value", unknown_value=-1` it becomes −1.
- **Ordinal plan:** "XL" is not in [S, M, L]: error, or −1 with the same option. Note −1 then means "smaller than S", which is wrong. Better to add XL to the category list when you know it exists.
- **One-hot:** with `handle_unknown="ignore"`, ES becomes all zeros.

In [2]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
g = pd.DataFrame({"country": ["ES"], "plan": ["XL"]})
lab = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(six[["country"]])
ordn = OrdinalEncoder(categories=[["S", "M", "L"]], handle_unknown="use_encoded_value", unknown_value=-1).fit(six[["plan"]])
oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(six[["country"]])
print("label:", lab.transform(g[["country"]]).ravel(), " ordinal:", ordn.transform(g[["plan"]]).ravel(),
      " one-hot:", oh.transform(g[["country"]]).ravel(), oh.get_feature_names_out())

label: [-1.]  ordinal: [-1.]  one-hot: [0. 0. 0.] ['country_DE' 'country_FR' 'country_UK']


## Exercise 2

In [3]:
g = ten["churned"].mean()
s = ten.groupby("country")["churned"].agg(["count", "mean"])
for m in [0, 2, 10]:
    s[f"m={m}"] = (s["count"] * s["mean"] + m * g) / (s["count"] + m)
s.round(3)

,count,mean,m=0,m=2,m=10
country,,,,,
DE,2,0.000,0.000,0.250,0.417
FR,3,0.667,0.667,0.600,0.538
NL,1,1.000,1.000,0.667,0.545
UK,4,0.500,0.500,0.500,0.500


$m = 0$ is the raw category mean (no smoothing). As $m$ grows every country is pulled towards the global 0.5; small groups (NL, one customer) move the most, large groups the least. $m$ is roughly "how many customers' worth of evidence must a category have before we believe its own rate".

## Exercise 3

In [4]:
from sklearn.preprocessing import TargetEncoder
wrong = TargetEncoder(random_state=0).fit(tr[["fav_product"]], y_tr)
a, t = fit_score(tr[base].assign(f=wrong.transform(tr[["fav_product"]]).ravel()),
                 te[base].assign(f=wrong.transform(te[["fav_product"]]).ravel()))
print(f"fit + transform on train: train {a:.1%}  test {t:.1%}")

right = TargetEncoder(random_state=0)
a, t = fit_score(tr[base].assign(f=right.fit_transform(tr[["fav_product"]], y_tr).ravel()),
                 te[base].assign(f=right.transform(te[["fav_product"]]).ravel()))
print(f"fit_transform on train:   train {a:.1%}  test {t:.1%}")

fit + transform on train: train 78.5%  test 62.4%
fit_transform on train:   train 67.8%  test 66.6%


`fit` then `transform` on the same rows encodes each training row with means that include its own label (smoothed, but still leaky), so training accuracy is inflated and the model leans on the feature. `fit_transform` cross-fits. Same class, different method, different result: read the documentation of anything that learns from the target.

## Exercise 4

In [5]:
from sklearn.tree import DecisionTreeClassifier
lab = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(tr[["country"]])
oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(tr[["country"]])
versions = {"label-encoded": (tr[base].assign(c=lab.transform(tr[["country"]]).ravel()), te[base].assign(c=lab.transform(te[["country"]]).ravel())),
            "one-hot": (np.hstack([tr[base], oh.transform(tr[["country"]])]), np.hstack([te[base], oh.transform(te[["country"]])]))}
for name, (A, B) in versions.items():
    t = DecisionTreeClassifier(max_depth=4, random_state=0).fit(A, y_tr)
    print(f"tree, {name:<14} test {t.score(B, y_te):.1%}")

tree, label-encoded  test 67.3%
tree, one-hot        test 66.7%


For a tree the two are about the same. A tree does not multiply codes by a slope; it splits ("code ≤ 30.5?"), and with enough splits it can isolate any group of codes. Label encoding is therefore acceptable for tree models (and much more compact for high-cardinality columns), while linear models, KNN and neural networks need one-hot or target encoding.